[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter5/vectara-ingest.ipynb)

# Chapter 5: Ingest Data into Vectara

Vectara is a managed RAG-as-a-service platform that handles chunking, embedding, indexing, and retrieval — so you can focus on your application logic rather than infrastructure. It supports both file uploads (where Vectara handles parsing) and structured document indexing (where you control the document hierarchy).

This notebook demonstrates both ingestion methods using the Vectara v2 API.

**What you'll learn:**
- Create a corpus in Vectara to organize your documents
- Upload files (PDF) for automatic parsing and indexing
- Index structured documents with custom sections and metadata

**Prerequisites:** Set `VECTARA_API_KEY` as an environment variable. Sign up at [vectara.com](https://vectara.com) for a free account.

## Upload file to Vectara

This section uploads a raw PDF file to Vectara, which automatically handles parsing, chunking, and embedding so you don't have to build that pipeline yourself.

In [1]:
import requests
import json
import os

Let's define the Vectara corpus key and API key

In [2]:
CORPUS_KEY = "RAGBOOK"
VECTARA_API_KEY = os.getenv("VECTARA_API_KEY")

# Create corpus if it doesn't exist
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "x-api-key": VECTARA_API_KEY
}

create_corpus_payload = {
    "key": CORPUS_KEY,
    "name": "RAG Book Examples",
    "description": "Corpus for Hands-on RAG book examples"
}

response = requests.post("https://api.vectara.io/v2/corpora", headers=headers, json=create_corpus_payload)
if response.status_code == 201:
    print(f"Created corpus '{CORPUS_KEY}'")
elif response.status_code == 409:
    print(f"Corpus '{CORPUS_KEY}' already exists")
else:
    print(f"Create corpus returned: {response.status_code} - {response.text}")

Corpus 'RAGBOOK' already exists


In [3]:
url = f"https://api.vectara.io/v2/corpora/{CORPUS_KEY}/upload_file"

payload={}
files=[
  ('file',
  ( 'pet_policy',
    open('pet_policy.pdf','rb'),
    'application/octet-stream')
  )
]
headers = {
  'Accept': 'application/json',
  'x-api-key': VECTARA_API_KEY
}

response = requests.post(url, headers=headers, data=payload, files=files)
res = response.json()
print(f"Upload status: {response.status_code}")
print(res)

Upload status: 201
{'id': 'pet_policy', 'metadata': {'Producer': 'Skia/PDF m118 Google Docs Renderer', 'Title': 'Employee Handbook - Company Pet Policy'}, 'storage_usage': {'bytes_used': 9215, 'metadata_bytes_used': 5803}}


## Index document into Vectara

As an alternative to file upload, you can index a pre-structured document with explicit sections and metadata, giving you fine-grained control over the document hierarchy.

In [4]:
url = f"https://api.vectara.io/v2/corpora/{CORPUS_KEY}/documents"

payload = json.dumps({
  "id": "selected-works-of-shakespeare",
  "type": "structured",
  "title": "William Shakespeare, Greatest Hits",
  "metadata": {
    "timespan": "26 April 1564---23 April 1616",
    "stars": 5,
    "author": "William Shakespeare"
  },
  "sections": [
    {
      "title": "King Lear",
      "text": "Synopsis: King Lear, intending to divide his power and kingdom among his three daughters, demands public professions of their love. His youngest daughter, ...",
      "sections": [
        {
          "title": "Act I",
          "text": "KENT: I thought the king had more affected the Duke of Albany than Cornwall.\nGLOUCESTER: It did always seem so to us...",
          "metadata": {
            "stage-instructions": "Enter KENT, GLOUCESTER, and EDMUND"
          }
        },
        {
          "title": "Act II",
          "text": "EDMUND: Save thee, Curan. ...",
          "metadata": {
            "stage-instructions": "Enter EDMUND, and CURAN meets him"
          }
        }
      ]
    },
    {
      "title": "Antony and Cleopatra",
      "text": "PHILO: Nay, but this dotage of our general's\nO'erflows the measure: those his goodly eyes, ..."
    }
  ]
})

headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'x-api-key': VECTARA_API_KEY
}

response = requests.post(url, headers=headers, data=payload)
res = response.json()
print(f"Index document status: {response.status_code}")
print(res)

Index document status: 412
{'messages': ['ALREADY_EXISTS: Document already processed.'], 'request_id': '3c889243da61eb48bb7f8eca24915e33'}
